In [ ]:
#@title 1. Settings and the declared thresholds
PROJECT_DIR     = "" #@param {type:"string"}
#@markdown The project folder. Left empty it falls back to DRIVE_ROOT and
#@markdown PROJECT_FOLDER below.
DRIVE_ROOT      = "/content/drive/MyDrive"  #@param {type:"string"}
PROJECT_FOLDER  = "KYNU truncated isoform" #@param {type:"string"}
HOLD_FRACTION   = 0.50   #@param {type:"number"}
#@markdown The mixed dimer passes the assembly arm when it keeps at least this
#@markdown fraction of the canonical buried interfacial area and contact count.
FAIL_FRACTION   = 0.25   #@param {type:"number"}
#@markdown Below this fraction the interface is gone and the series stops. Between the
#@markdown two the assembly arm is inconclusive and needs longer sampling.
SEPARATED_A     = 5.0    #@param {type:"number"}
#@markdown A shortest inter-subunit distance above this over the last tenth means the
#@markdown subunits are no longer in contact at all.
EXPOSURE_FRACTION = 0.50 #@param {type:"number"}
#@markdown The pocket arm passes when the cofactor in the mixed dimer is more exposed
#@markdown than in the canonical by at least this fraction of the crystallographic
#@markdown partner burial, which comes from the build spec.
TAIL_FRACTION   = 0.10   #@param {type:"number"}
#@markdown The last tenth of each trajectory is scored on its own, so an interface
#@markdown still coming apart at the end does not pass on its average.
MIN_NS_FOR_HOLD = 100.0  #@param {type:"number"}
#@markdown The criterion is one-sided below this length.
#@markdown
#@markdown Whether a cofactor is still in the site it started in is part 3b's call,
#@markdown taken from the `settled` flag it writes per build, so the displacement that
#@markdown counts as seated is set in one place and not again here.

import os
PROJECT = PROJECT_DIR.strip() or os.path.join(DRIVE_ROOT, PROJECT_FOLDER)
MD = os.path.join(PROJECT, "md")
INPUTS = os.path.join(MD, "inputs")
RESULTS = os.path.join(MD, "results")
FIGURES = os.path.join(PROJECT, "figures")
BUILD_ORDER = ["canonical", "mixed"]
print("project    ", PROJECT)
print("assembly    hold at or above %.2f of canonical, stop below %.2f, separated "
      "above %.1f A" % (HOLD_FRACTION, FAIL_FRACTION, SEPARATED_A))
print("pocket      a rise of at least %.2f of the crystallographic partner burial"
      % EXPOSURE_FRACTION)


In [ ]:
#@title 2. Read both runs and apply the criterion
from google.colab import drive
drive.mount('/content/drive')

import collections, datetime, glob, json, os, sys
import numpy as np

os.makedirs(FIGURES, exist_ok=True)


def fatal(msg):
    raise SystemExit("STOP: " + msg)


SPEC_PATH = os.path.join(INPUTS, "build_spec.json")
if not os.path.isfile(SPEC_PATH):
    fatal("no %s. Part 1 writes it and this notebook reads the crystallographic "
          "burial from it rather than carrying its own copy." % SPEC_PATH)
SPEC = json.load(open(SPEC_PATH))
SOURCE = SPEC["cofactor_source"]
CRYSTAL = SPEC["crystal_partner_burial_A2"]
if SOURCE not in CRYSTAL:
    fatal("the spec says the cofactor pose came from %s and its burial table holds "
          "%s" % (SOURCE, sorted(CRYSTAL)))
LOST_BURIAL = float(CRYSTAL[SOURCE]["lost_block"])
WHOLE_BURIAL = float(CRYSTAL[SOURCE]["whole_partner"])
if LOST_BURIAL <= 0:
    fatal("the crystal measurement puts %.2f A2 of the cofactor burial in the span "
          "the isoform lacks, so there is no predicted rise to test for" % LOST_BURIAL)
RISE_WANTED = EXPOSURE_FRACTION * LOST_BURIAL
print("crystal %s: the partner buries %.2f A2 of the cofactor, %.2f of it from the "
      "span the isoform lacks" % (SOURCE, WHOLE_BURIAL, LOST_BURIAL))
print("pocket arm asks for a rise of at least %.2f A2" % RISE_WANTED)

REP = {}
for b in BUILD_ORDER:
    p = os.path.join(RESULTS, "%s_report.json" % b)
    if not os.path.isfile(p):
        fatal("no %s. Run the part 2 notebook for the %s build first; this one "
              "compares the two and cannot do it from one." % (p, b))
    REP[b] = json.load(open(p))

for b in BUILD_ORDER:
    if REP[b]["frames"] < 100:
        fatal("the %s run holds %d frames, which is too few to score"
              % (b, REP[b]["frames"]))
ns = {b: REP[b]["ns"] for b in BUILD_ORDER}
if abs(ns["canonical"] - ns["mixed"]) > 0.05 * max(ns.values()):
    fatal("the two runs span %s ns, and the comparison needs matched sampling. Let "
          "the shorter one catch up before scoring." % ns)
SHORT = min(ns.values()) < MIN_NS_FOR_HOLD
if REP["canonical"]["cofactor"] != REP["mixed"]["cofactor"]:
    fatal("the two runs carry different cofactors, %s and %s"
          % (REP["canonical"]["cofactor"], REP["mixed"]["cofactor"]))
print("both runs span %.0f ns, %d and %d frames"
      % (ns["canonical"], REP["canonical"]["frames"], REP["mixed"]["frames"]))

DRIFT_PATH = os.path.join(RESULTS, "cofactor_drift.json")
if not os.path.isfile(DRIFT_PATH):
    fatal("no %s. Run kynu_MD_3b_cofactor_drift.ipynb first. It measures how far the "
          "cofactor has moved inside its own subunit, and without that the pocket arm "
          "cannot tell a pocket that opened from a cofactor that left." % DRIFT_PATH)
DRIFT = json.load(open(DRIFT_PATH))
missing = [b for b in BUILD_ORDER if b not in DRIFT]
if missing:
    fatal("%s covers %s and this needs %s. Run part 3b again after both builds have "
          "finished." % (DRIFT_PATH, sorted(DRIFT), missing))
for b in BUILD_ORDER:
    if abs(DRIFT[b].get("ns", 0) - ns[b]) > 0.05 * max(ns.values()):
        fatal("the %s drift measurement spans %.0f ns and its run spans %.0f. Run "
              "part 3b again so the two describe the same trajectory."
              % (b, DRIFT[b].get("ns", 0), ns[b]))
unflagged = [b for b in BUILD_ORDER if "settled" not in DRIFT[b]]
if unflagged:
    fatal("%s carries no settled flag for %s, so it was written by an earlier part 3b. "
          "Run kynu_MD_3b_cofactor_drift.ipynb again." % (DRIFT_PATH, unflagged))
WANDERED = sorted(b for b in BUILD_ORDER if not DRIFT[b]["settled"])
POCKET_EVALUABLE = not WANDERED
for b in BUILD_ORDER:
    print("  %-10s cofactor sits %.2f +/- %.2f A from where its run started, %.2f over "
          "the first tenth and %.2f over the last"
          % (b, DRIFT[b]["mean"], DRIFT[b]["sd"], DRIFT[b]["first_tenth"],
             DRIFT[b]["last_tenth"]))
if WANDERED:
    print("  the cofactor left its starting site in %s, so the pocket arm is not "
          "scored" % " and ".join(WANDERED))


def series(build, name, column=1, scale=1.0):
    base = REP[build].get("run")
    if not base:
        fatal("the %s report does not say which folder it ran from, so it was written "
              "by an earlier part 2. Re-run part 2 for that build." % build)
    p = os.path.join(base, name)
    if not os.path.isfile(p):
        fatal("no %s. The part 2 run stages every xvg it writes to Drive, so a "
              "missing one means that staging did not finish." % p)
    D = np.loadtxt(p, comments=["#", "@"])
    if D.ndim == 1:
        D = D.reshape(1, -1)
    return D[:, 0], D[:, column] * scale


def buried_series(build):
    t, ab = series(build, "sasa_dimer.xvg")
    _, a = series(build, "sasa_sub1.xvg")
    _, b = series(build, "sasa_sub2.xvg")
    n = min(len(ab), len(a), len(b))
    return t[:n], (a[:n] + b[:n] - ab[:n]) * 100.0 / 2.0


def cofactor_series(build):
    return series(build, "sasa_cofactor.xvg", column=2, scale=100.0)


def scored(v, burn, tail=False):
    v = np.asarray(v, dtype=float)
    w = v[int(len(v) * burn):]
    if tail:
        w = w[-max(1, int(len(w) * TAIL_FRACTION)):]
    return w


BURN = REP["canonical"]["burn"] / float(REP["canonical"]["frames"])
S = {}
for b in BUILD_ORDER:
    t, bur = buried_series(b)
    _, cof = cofactor_series(b)
    _, nc = series(b, "numcont_dimer.xvg")
    _, md = series(b, "mindist_dimer.xvg", scale=10.0)
    n = min(len(t), len(bur), len(cof), len(nc), len(md))
    S[b] = dict(t=t[:n], buried=bur[:n], cofactor=cof[:n], contacts=nc[:n],
                mindist=md[:n])
    print("  %-10s cofactor %5.1f A2, buried %6.0f A2 per subunit, over the scored "
          "window" % (b, scored(S[b]["cofactor"], BURN).mean(),
                      scored(S[b]["buried"], BURN).mean()))

ratio = {}
for key in ("buried", "contacts"):
    can = scored(S["canonical"][key], BURN).mean()
    mix = scored(S["mixed"][key], BURN).mean()
    can_tail = scored(S["canonical"][key], BURN, tail=True).mean()
    mix_tail = scored(S["mixed"][key], BURN, tail=True).mean()
    if can <= 0:
        fatal("the canonical run gives %s of %g, so there is nothing to compare "
              "against" % (key, can))
    ratio[key] = dict(canonical=float(can), mixed=float(mix), fraction=float(mix / can),
                      canonical_tail=float(can_tail), mixed_tail=float(mix_tail),
                      fraction_tail=float(mix_tail / can_tail) if can_tail > 0 else 0.0)
    print("  %-9s mixed keeps %.2f of canonical over the window and %.2f over the "
          "last tenth" % (key, ratio[key]["fraction"], ratio[key]["fraction_tail"]))

cof_can = scored(S["canonical"]["cofactor"], BURN)
cof_mix = scored(S["mixed"]["cofactor"], BURN)
rise = float(cof_mix.mean() - cof_can.mean())
pocket = dict(canonical_A2=float(cof_can.mean()), canonical_sd=float(cof_can.std()),
              mixed_A2=float(cof_mix.mean()), mixed_sd=float(cof_mix.std()),
              rise_A2=rise, rise_wanted_A2=float(RISE_WANTED),
              crystal_lost_block_A2=LOST_BURIAL,
              crystal_whole_partner_A2=WHOLE_BURIAL,
              fraction_of_crystal=float(rise / LOST_BURIAL))
print("  cofactor exposure %.1f A2 canonical against %.1f A2 mixed, a rise of %.1f "
      "A2, %.2f of the crystallographic %.2f A2"
      % (pocket["canonical_A2"], pocket["mixed_A2"], rise,
         pocket["fraction_of_crystal"], LOST_BURIAL))

sep = float(scored(S["mixed"]["mindist"], BURN, tail=True).mean())
worst = min(ratio[k]["fraction"] for k in ratio)
worst_tail = min(ratio[k]["fraction_tail"] for k in ratio)
assembly_holds = worst >= HOLD_FRACTION and worst_tail >= HOLD_FRACTION
assembly_fails = (sep > SEPARATED_A or worst < FAIL_FRACTION
                  or worst_tail < FAIL_FRACTION)
pocket_passes = POCKET_EVALUABLE and rise >= RISE_WANTED

if assembly_fails:
    if sep > SEPARATED_A:
        verdict = "STOP"
        why = ("the mixed subunits are %.1f A apart over the last tenth of the run, "
               "so they are no longer in contact and the isoform is not competing for "
               "the dimer at all" % sep)
    else:
        verdict = "STOP"
        why = ("the mixed dimer keeps %.2f of the canonical interface over the window "
               "and %.2f over its last tenth, below the declared %.2f, so it does not "
               "hold together long enough to compete"
               % (worst, worst_tail, FAIL_FRACTION))
elif SHORT:
    verdict = "CONTINUE"
    why = ("the runs span %.0f ns and nothing has fired a stop arm. The mixed dimer "
           "keeps %.2f of the canonical interface and the cofactor has risen %.1f A2 "
           "of the %.1f asked for, and a pass is withheld until %.0f ns, so carry "
           "both runs on." % (min(ns.values()), worst, rise, RISE_WANTED,
                              MIN_NS_FOR_HOLD))
elif assembly_holds and not POCKET_EVALUABLE:
    verdict = "ASSEMBLY HOLDS, POCKET NOT EVALUABLE"
    why = ("the mixed dimer keeps %.2f of the canonical interface over the window and "
           "%.2f over its last tenth, at or above the declared %.2f, so the assembly "
           "arm passes. The pocket arm is not scored: the cofactor moved %s from "
           "where its run started, which part 3b scores as having left the site, "
           "so the exposure measured in each build is of a cofactor in an "
           "undetermined position and the difference between them says nothing about "
           "the partner's contribution to the site."
           % (worst, worst_tail, HOLD_FRACTION,
              " and ".join("%.2f A in %s" % (DRIFT[b]["mean"], b)
                           for b in WANDERED)))
elif assembly_holds and pocket_passes:
    verdict = "SUPPORTS"
    why = ("the mixed dimer keeps %.2f of the canonical interface over the window and "
           "%.2f over its last tenth, at or above the declared %.2f, and the cofactor "
           "in its canonical subunit is %.1f A2 more exposed than in the canonical "
           "homodimer, which is %.2f of the crystallographic burial and at or above "
           "the declared %.2f" % (worst, worst_tail, HOLD_FRACTION, rise,
                                  pocket["fraction_of_crystal"], EXPOSURE_FRACTION))
elif assembly_holds and not pocket_passes:
    verdict = "NO DETECTABLE POCKET EFFECT"
    why = ("the mixed dimer holds its interface, keeping %.2f of the canonical, and "
           "the cofactor in its canonical subunit is %.1f A2 more exposed than in the "
           "canonical homodimer, short of the %.1f A2 asked for. Over this sampling "
           "the exposure the crystal measurement predicts is not detected, which "
           "bounds the effect at this length and does not show that there is none."
           % (worst, rise, RISE_WANTED))
else:
    verdict = "INCONCLUSIVE"
    why = ("the mixed dimer keeps %.2f of the canonical interface over the window and "
           "%.2f over its last tenth, between the declared %.2f and %.2f, so the "
           "assembly arm needs longer sampling or replicates instead of a verdict"
           % (worst, worst_tail, FAIL_FRACTION, HOLD_FRACTION))

out = dict(verdict=verdict, reason=why,
           thresholds=dict(hold=HOLD_FRACTION, stop=FAIL_FRACTION,
                           separated_A=SEPARATED_A, exposure_fraction=EXPOSURE_FRACTION,
                           tail_fraction=TAIL_FRACTION, burn_fraction=BURN,
                           min_ns_for_hold=MIN_NS_FOR_HOLD),
           one_sided=bool(SHORT),
           assembly=dict(holds=bool(assembly_holds), fails=bool(assembly_fails),
                         interface=ratio, mixed_mindist_tail_A=sep),
           pocket=pocket, pocket_evaluable=bool(POCKET_EVALUABLE),
           cofactor_drift=DRIFT,
           ns=ns, frames={b: REP[b]["frames"] for b in BUILD_ORDER},
           rmsd_A={b: REP[b]["rmsd_A"] for b in BUILD_ORDER},
           mindist_cofactor_partner_A={b: REP[b]["mindist_cofactor_partner_A"]
                                       for b in BUILD_ORDER},
           cofactor=REP["canonical"]["cofactor"], cofactor_source=SOURCE,
           written=datetime.datetime.now().isoformat(timespec="seconds"))
json.dump(out, open(os.path.join(RESULTS, "kill_criterion.json"), "w"), indent=2)

print("\n" + "=" * 72)
print("CRITERION: %s" % verdict)
print("=" * 72)
print(why)
print("\nbackbone RMSD: canonical %.2f +/- %.2f A, mixed %.2f +/- %.2f A"
      % (REP["canonical"]["rmsd_A"]["mean"], REP["canonical"]["rmsd_A"]["sd"],
         REP["mixed"]["rmsd_A"]["mean"], REP["mixed"]["rmsd_A"]["sd"]))
print("shortest cofactor to partner distance: canonical %.2f A, mixed %.2f A"
      % (REP["canonical"]["mindist_cofactor_partner_A"]["mean"],
         REP["mixed"]["mindist_cofactor_partner_A"]["mean"]))
print("\nwrote %s" % os.path.join(RESULTS, "kill_criterion.json"))
if verdict == "CONTINUE":
    print("\nRaise NS_PER_BUILD in both part 2 notebooks to %.0f and press Run all "
          "again. The finished chunks are skipped." % MIN_NS_FOR_HOLD)


In [ ]:
#@title 3. The figure
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

COL = {"canonical": "#4C6EF5", "mixed": "#E8590C"}
fig, ax = plt.subplots(1, 4, figsize=(17.5, 4.0), facecolor="white")
panels = (("cofactor", "Cofactor solvent accessible surface, $\\mathrm{\\AA}^2$"),
          ("buried", "Buried interfacial area per subunit, $\\mathrm{\\AA}^2$"),
          ("contacts", "Inter-subunit contacts within 5 $\\mathrm{\\AA}$"),
          ("mindist", "Shortest inter-subunit distance, $\\mathrm{\\AA}$"))
for a, (key, label) in zip(ax, panels):
    ends = []
    for b in BUILD_ORDER:
        t = S[b]["t"] / 1000.0
        v = S[b][key]
        n = min(len(t), len(v))
        a.plot(t[:n], v[:n], lw=0.8, color=COL[b], label=b)
        ends.append((b, float(t[n - 1]), float(v[n - 1])))
    ends.sort(key=lambda e: e[2])
    span = max(e[2] for e in ends) - min(e[2] for e in ends)
    for i, (b, x, y) in enumerate(ends):
        dy = 0 if span > 0.02 * (abs(y) + 1e-9) else (-7 + 14 * i)
        a.annotate(b, xy=(x, y), xytext=(4, dy), textcoords="offset points",
                   color=COL[b], fontsize=9, va="center")
    a.set_xlabel("time, ns")
    a.set_ylabel(label)
    a.set_facecolor("white")
    for s in ("top", "right"):
        a.spines[s].set_visible(False)
if POCKET_EVALUABLE:
    target = pocket["canonical_A2"] + RISE_WANTED
    ax[0].axhline(target, ls="--", lw=0.8, color="#868E96")
    ax[0].annotate("declared rise, %.2f of the crystallographic %.1f "
                   "$\\mathrm{\\AA}^2$" % (EXPOSURE_FRACTION, LOST_BURIAL),
                   xy=(0.02, target), xytext=(0, 4), textcoords="offset points",
                   fontsize=8, color="#868E96")
else:
    ax[0].annotate("not scored: the cofactor moved %s from its starting pose"
                   % " and ".join("%.1f $\\mathrm{\\AA}$ in %s"
                                  % (DRIFT[b]["mean"], b) for b in WANDERED),
                   xy=(0.02, 0.02), xycoords="axes fraction", fontsize=8,
                   color="#868E96")
ax[1].axhline(ratio["buried"]["canonical"] * HOLD_FRACTION, ls="--", lw=0.8,
              color="#868E96")
ax[1].annotate("declared hold threshold, %.2f of canonical" % HOLD_FRACTION,
               xy=(0.02, ratio["buried"]["canonical"] * HOLD_FRACTION),
               xytext=(0, 4), textcoords="offset points", fontsize=8, color="#868E96")
fig.suptitle("KYNU truncated isoform, %s, %.0f ns each" % (out["verdict"],
                                                           ns["canonical"]),
             y=1.02, fontsize=11)
fig.tight_layout()
for ext in ("png", "svg"):
    p = os.path.join(FIGURES, "md_criterion.%s" % ext)
    fig.savefig(p, dpi=400, facecolor="white", bbox_inches="tight")
    print("wrote", p)
plt.close(fig)
